In [58]:
import numpy as np
import requests
import json
import pprint
import polars as pl
from polars import col
import joblib

pl.Config.set_tbl_cols(n = 15)
pl.Config.set_tbl_rows(n = 25)
pl.Config.set_tbl_width_chars(300)

polars.config.Config

In [59]:
# Prameters for the API request
params = {
    "latitude": 23.7106,        # for Dhaka. 
    "longitude": 90.4067,       # for Dhaka. 
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m", # unit : %.
        "pressure_msl",         # unit : hPa. Sealevel pressure.
        "cloud_cover_low",      # unit : %. The most important cloud layer for rain.
        "cloud_cover",    # unit : %. Great overall context.
        "vapour_pressure_deficit", # unit : kPa (kilopascal).
        # "visibility",           # unit : meters. It's 'how far you can see clearly through the air' which changes frequently for
        "wind_speed_10m",         # very weather conditions INSTANT. Its a real-time sensor for CURRENT DATA, not for Historical.
        "wind_direction_10m",   # unit : °.
        "wind_gusts_10m",       # unit : km/h.
        "precipitation",              # precipitation = the amount of water that is expected to fall from the sky.
        "precipitation_probability" # precipitation_probability = What's the chance it'll fall? That means its for FORECASTING,
    ],                                # not what already happened in the past i.e. Historical Data. Setting it will return NULLs.
    # "daily": "temperature_2m_max,temperature_2m_min,wind_speed_10m_max,wind_gusts_10m_max",
    "timezone": "Asia/Dhaka", # Set timezone to Dhaka to get 24 hours data aligned with Dhaka's Local time from 0 to 23 because
    # The data is for Dhaka, so timezone ofc need to be aligned with Dhaka's local time, not with another city/country's timezone.
    # units of the parameters whose units are not constant :
    "temperature_unit" : "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
    "forecast_hours": 7
}

# Make the API request.
url = "https://api.open-meteo.com/v1/forecast"
response = requests.get(url, params = params)
data = response.json()

print(json.dumps(data, indent = 4))

{
    "latitude": 23.75,
    "longitude": 90.375,
    "generationtime_ms": 0.12314319610595703,
    "utc_offset_seconds": 21600,
    "timezone": "Asia/Dhaka",
    "timezone_abbreviation": "GMT+6",
    "elevation": 27.0,
    "hourly_units": {
        "time": "iso8601",
        "temperature_2m": "\u00b0C",
        "relative_humidity_2m": "%",
        "pressure_msl": "hPa",
        "cloud_cover_low": "%",
        "cloud_cover": "%",
        "vapour_pressure_deficit": "kPa",
        "wind_speed_10m": "km/h",
        "wind_direction_10m": "\u00b0",
        "wind_gusts_10m": "km/h",
        "precipitation": "mm",
        "precipitation_probability": "%"
    },
    "hourly": {
        "time": [
            "2025-09-09T02:00",
            "2025-09-09T03:00",
            "2025-09-09T04:00",
            "2025-09-09T05:00",
            "2025-09-09T06:00",
            "2025-09-09T07:00",
            "2025-09-09T08:00"
        ],
        "temperature_2m": [
            28.0,
            27.9,
     

In [75]:
hourly_data: dict = data.get("hourly", {}) # dict.get(key, default).
pprint.PrettyPrinter(width = 2).pprint(list(hourly_data.keys())) # Just to print vertically.
type(hourly_data['temperature_2m'])

['time',
 'temperature_2m',
 'relative_humidity_2m',
 'pressure_msl',
 'cloud_cover_low',
 'cloud_cover',
 'vapour_pressure_deficit',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'precipitation',
 'precipitation_probability']


list

In [63]:
#                  Convert the dict/json data to dataframe with CORRECT TIME ZONE and Feature Engineering.

df = ( pl.DataFrame(data = hourly_data)
      .lazy()
      .rename({"time" : "date"}) # {old_name : new_name}.
      .with_columns(date  = col("date").str.to_datetime(format = "%Y-%m-%dT%H:%M", time_zone = data['timezone']))
      .with_columns(col("relative_humidity_2m", "cloud_cover_low", "cloud_cover").cast(pl.Int8),
                    col("wind_direction_10m")                                    .cast(pl.Int16),

                    hour        = col('date').dt.hour(),  # 1 to 24 hours.
                    month       = col('date').dt.month(), # 1 to 12 no month.
                    day_of_year = col('date').dt.ordinal_day(), # 1 to 366 days.
                    rainfall    = ( pl.when(col("precipitation") <= 0)
                                          .then(0)  # Clear Sky.
                                    .when(col("precipitation") < 2.1)
                                          .then(1)  # Light rain.
                                    .when(col("precipitation") < 10.1)
                                          .then(2)  # Noticeable rain.
                                    .when(col("precipitation") < 30.1)
                                          .then(3)  # Heavy rain.
                                    .when(col("precipitation") < 50.1)
                                          .then(4)  # Very heavy rain.
                                    .otherwise(5) ).cast(pl.Int8), # Extreme rainfall.
                    date_am_pm  = col('date').dt.strftime(format = "%d %B %Y : %I %p")
      )
      .drop("date", "precipitation", "precipitation_probability")
      .select(col("hour", "month", "day_of_year"),
              col('*').exclude("hour", "month", "day_of_year")) # Features at the front + Target column at the end.
      .collect()
)

print(df)

shape: (7, 14)
┌──────┬───────┬─────────────┬────────────────┬──────────────────────┬──────────────┬─────────────────┬─────────────┬─────────────────────────┬────────────────┬────────────────────┬────────────────┬──────────┬───────────────────────────┐
│ hour ┆ month ┆ day_of_year ┆ temperature_2m ┆ relative_humidity_2m ┆ pressure_msl ┆ cloud_cover_low ┆ cloud_cover ┆ vapour_pressure_deficit ┆ wind_speed_10m ┆ wind_direction_10m ┆ wind_gusts_10m ┆ rainfall ┆ date_am_pm                │
│ ---  ┆ ---   ┆ ---         ┆ ---            ┆ ---                  ┆ ---          ┆ ---             ┆ ---         ┆ ---                     ┆ ---            ┆ ---                ┆ ---            ┆ ---      ┆ ---                       │
│ i8   ┆ i8    ┆ i16         ┆ f64            ┆ i8                   ┆ f64          ┆ i8              ┆ i8          ┆ f64                     ┆ f64            ┆ i16                ┆ f64            ┆ i8       ┆ str                       │
╞══════╪═══════╪═════════════╪═══

In [78]:
rainfall_probability = hourly_data['precipitation_probability']
rainfall = df['rainfall'].to_numpy()

X_test = df.drop('rainfall', 'date_am_pm').to_pandas()
print(X_test)
print("\n", rainfall)

   hour  month  day_of_year  temperature_2m  relative_humidity_2m  \
0     2      9          252            28.0                    92   
1     3      9          252            27.9                    92   
2     4      9          252            27.8                    91   
3     5      9          252            27.6                    91   
4     6      9          252            27.6                    92   
5     7      9          252            27.8                    92   
6     8      9          252            28.1                    89   

   pressure_msl  cloud_cover_low  cloud_cover  vapour_pressure_deficit  \
0        1005.4               33          100                     0.30   
1        1005.4               37          100                     0.30   
2        1005.3               36          100                     0.33   
3        1005.5               44          100                     0.33   
4        1006.1               49          100                     0.29   
5  

In [79]:
weather_model = joblib.load(filename = "weather_model.pkl")
print(f"The XGB Model predicted : {weather_model.predict(X_test)}.")
print(f"The OpenMeteo predicted : {rainfall}.")

The XGB Model predicted : [1 1 1 1 0 1 0].
The OpenMeteo predicted : [0 0 0 0 1 1 1].


# Time Zone and Lat, long from English Address.

In [70]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent = "my_geocoder_app")
location = geolocator.geocode("Narayanganj, Bangladesh")

if location:
    print(f"Address: {location.address}")
    print(f"Latitude: {location.latitude}")
    print(f"Longitude: {location.longitude}")
else:
    print("Location not found.")

Address: নারায়ণগঞ্জ, নারায়ণগঞ্জ সদর উপজেলা, নারায়নগঞ্জ জেলা, ঢাকা বিভাগ, 1400, বাংলাদেশ
Latitude: 23.6236732
Longitude: 90.4988075


In [73]:
from geopy.geocoders import Nominatim
from tzfpy import get_tz, get_tzs

# Step 1: Geocode the address
geolocator = Nominatim(user_agent="timezone_finder_app")
address = "Eiffel Tower, Paris, France"
location = geolocator.geocode(address)

if location:
    print(location)
    latitude = location.latitude
    longitude = location.longitude
    print(f"Coordinates for '{address}': Latitude={latitude}, Longitude={longitude}")

    # Step 2: Determine timezone from coordinates
    timezone_name = get_tz(longitude, latitude)
    print(f"Timezone for these coordinates: {timezone_name}")
else:
    print(f"Could not find coordinates for '{address}'")

Tour Eiffel, 5, Avenue Anatole France, Quartier du Gros-Caillou, Paris 7e Arrondissement, Paris, France métropolitaine, 75007, France
Coordinates for 'Eiffel Tower, Paris, France': Latitude=48.8582599, Longitude=2.2945006
Timezone for these coordinates: Europe/Paris


In [ ]:
def rainfall_convertion(arr):
    effects  = np.array(['Clear Sky', 'Light rain', 'Noticeable rain', 'Heavy rain', 'Very heavy rain'])
    return effects[arr] # fancy indexing.

def current_weather_info_for_next_5_hours(city_name: str):
    # 1) Finding latitude and longitude of the city name.
    city_name = f"{city_name}, Bangladesh"

    geolocator = Nominatim(user_agent = "my_geocoder_app")
    location = geolocator.geocode(city_name)

    # 2) Prameters for the API request.
    params = {
        "latitude": location.latitude,
        "longitude": location.longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m", # unit : %.
            "pressure_msl",         # unit : hPa. Sealevel pressure.
            "cloud_cover_low",      # unit : %. The most important cloud layer for rain.
            "cloud_cover",    # unit : %. Great overall context.
            "vapour_pressure_deficit", # unit : kPa (kilopascal).
            # "visibility",           # unit : meters.
            "wind_speed_10m",         
            "wind_direction_10m",   # unit : °.
            "wind_gusts_10m",       # unit : km/h.
            # "precipitation",              # precipitation = the amount of water that is expected to fall from the sky.
            # "precipitation_probability" # precipitation_probability = What's the chance it'll fall? That means its for FORECASTING,
        ],
        "timezone": "Asia/Dhaka", # Set timezone to Dhaka to get 24 hours data aligned with Dhaka's Local time from 0 to 23 because
        "temperature_unit" : "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm",
        "forecast_hours": 6 # current hour + next 5 hours.
    }

    # Make the API request.
    url = "https://api.open-meteo.com/v1/forecast"
    response = requests.get(url, params = params)
    data = response.json()

    hourly_data: dict = data.get("hourly", {}) # dict.get(key, default).

    # 3) Convert the dict/json data to dataframe with CORRECT TIME ZONE and Feature Engineering.
    df = ( pl.DataFrame(data = hourly_data)
        .lazy()
        .rename({"time" : "date"}) # {old_name : new_name}.
        .with_columns(date  = col("date").str.to_datetime(format = "%Y-%m-%dT%H:%M", time_zone = data['timezone']))
        .with_columns(  col("relative_humidity_2m", "cloud_cover_low", "cloud_cover").cast(pl.Int8),
                        col("wind_direction_10m")                                    .cast(pl.Int16),

                        hour        = col('date').dt.hour(),  # 1 to 24 hours.
                        month       = col('date').dt.month(), # 1 to 12 no month.
                        day_of_year = col('date').dt.ordinal_day(), # 1 to 366 days.
                        date_am_pm  = col('date').dt.strftime(format = "%d %B %Y %I %p"),
        )
        .drop("date")
        .select(col("hour", "month", "day_of_year"),
                col('*').exclude("hour", "month", "day_of_year")) # Features at the front + Target column at the end.
        .collect()
        .to_pandas()
    )

    # 4) Create X_test data and make prediction.
    dates = df['date_am_pm'] # Series.
    X_test = df.drop(columns = 'date_am_pm') # Dataframe.

    print(X_test)
    prediction = rainfall_convertion(weather_model.predict(X_test)) # numpy 1D array.
    
    # 5) return the necessery info for all the 6 cards.
    return dates, df['temperature_2m'], prediction

In [89]:
dates, temperature, prediction =  current_weather_info_for_next_5_hours("Dhaka")

print(dates)
print("\n", temperature)
print("\n", prediction)

   hour  month  day_of_year  temperature_2m  relative_humidity_2m  \
0     7      9          252            28.3                    91   
1     8      9          252            29.0                    88   
2     9      9          252            29.3                    86   
3    10      9          252            30.3                    82   
4    11      9          252            31.1                    78   
5    12      9          252            31.8                    74   

   pressure_msl  cloud_cover_low  cloud_cover  vapour_pressure_deficit  \
0        1006.6               40           92                     0.35   
1        1007.6               68          100                     0.48   
2        1008.2               76          100                     0.57   
3        1008.4               77          100                     0.77   
4        1007.9               81          100                     0.99   
5        1006.9               74          100                     1.22  